RAG Pipeline - Data ingestion to vector db

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
#Read all the pdf's in a directory

def process_all_pdfs(pdf_directory):
    """Process all the pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source metadata to each document
            for doc in documents:
                doc.metadata["source"] = str(pdf_file)  
                doc.metadata["fileType"] = "pdf"
            
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Processed {len(all_documents)} documents from {len(pdf_files)} PDF files.")
    return all_documents

#Process all pdf's in the data/pdf directory
pdf_directory = "../data/pdf"
all_documents = process_all_pdfs(pdf_directory)

In [ ]:
all_documents

In [ ]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #show example of a chunk
    if split_docs:
        print(f"Example chunk: {split_docs[0].page_content[:500]}...")  # Print first 500 characters of the first chunk
        print(f"Metadata of the first chunk: {split_docs[0].metadata}")
        
    return split_docs